# GOLD LAYER

Dashboard
1. Sales and Revenue
2. Sellers and Operations
3. Products and Customer Experience



In [18]:

import duckdb
import os
import pandas as pd

con = duckdb.connect('../olist.duckdb')

In [19]:
# ORDER-LEVEL TABLES
# --- prevent double-counting


# ORDER REVIEWS

con.execute("""
CREATE OR REPLACE TABLE gold_order_reviews AS

SELECT
    order_id,

    AVG(review_score) AS review_score,

    COUNT(*) AS review_count,

    MIN(review_creation_date) AS review_creation_date

FROM silver_order_reviews

GROUP BY order_id
""")

print("Created gold_order_reviews.")


# ORDER PAYMENTS

con.execute("""
CREATE OR REPLACE TABLE gold_order_payments AS

SELECT
    order_id,

    SUM(payment_value) AS total_payment_value,

    SUM(
        CASE
            WHEN payment_type = 'credit_card'
            THEN payment_value
            ELSE 0
        END
    ) AS credit_card_payment_value,

    SUM(
        CASE
            WHEN payment_type = 'boleto'
            THEN payment_value
            ELSE 0
        END
    ) AS boleto_payment_value,

    SUM(
        CASE
            WHEN payment_type = 'voucher'
            THEN payment_value
            ELSE 0
        END
    ) AS voucher_payment_value,

    SUM(
        CASE
            WHEN payment_type = 'debit_card'
            THEN payment_value
            ELSE 0
        END
    ) AS debit_card_payment_value

FROM silver_payments

GROUP BY order_id
""")

print("Created gold_order_payments.")


# ORDER ITEMS

con.execute("""
CREATE OR REPLACE TABLE gold_order_items AS

SELECT
    order_id,

    COUNT(*) AS item_count,

    SUM(price) AS total_sales,

    SUM(freight_value) AS total_freight,

    SUM(price + freight_value) AS sales_plus_freight

FROM silver_order_items

GROUP BY order_id
""")

print("Created gold_order_items.")


Created gold_order_reviews.
Created gold_order_payments.
Created gold_order_items.


In [20]:
# Sales and Revenue


# MONTHLY SALES

con.execute("""
CREATE OR REPLACE TABLE gold_sales_monthly_2017 AS

SELECT

    DATE_TRUNC(
        'month',
        o.order_purchase_timestamp
    ) AS month_date,

    STRFTIME(
        o.order_purchase_timestamp,
        '%Y-%m'
    ) AS month,

    STRFTIME(
        o.order_purchase_timestamp,
        '%B'
    ) AS month_name,

    COUNT(DISTINCT o.order_id) AS order_count,

    SUM(oi.price) AS total_sales,

    SUM(oi.freight_value) AS total_freight,

    SUM(oi.price + oi.freight_value)
        AS sales_plus_freight

FROM silver_orders o

INNER JOIN silver_order_items oi
    ON o.order_id = oi.order_id

WHERE
    o.order_purchase_timestamp >= '2017-01-01'

    AND o.order_purchase_timestamp < '2018-01-01'

GROUP BY
    month_date,
    month,
    month_name

ORDER BY
    month_date
""")

print("Created gold_sales_monthly_2017.")


# HIGHEST SALES MONTH

con.execute("""
CREATE OR REPLACE TABLE gold_sales_peak_2017 AS

SELECT

    month_date,

    month,

    month_name,

    total_sales,

    order_count

FROM gold_sales_monthly_2017

ORDER BY total_sales DESC

LIMIT 1
""")

print("Created gold_sales_peak_2017.")


# CREDIT CARD MONTHLY RUNNING TOTAL — 2017

con.execute("""
CREATE OR REPLACE TABLE gold_credit_card_running_total_2017 AS

WITH monthly_credit_card AS (

    SELECT

        DATE_TRUNC(
            'month',
            o.order_purchase_timestamp
        ) AS month_date,

        STRFTIME(
            o.order_purchase_timestamp,
            '%Y-%m'
        ) AS month,

        SUM(
            CASE
                WHEN p.payment_type = 'credit_card'
                THEN p.payment_value
                ELSE 0
            END
        ) AS monthly_credit_card_volume

    FROM silver_orders o

    INNER JOIN silver_payments p
        ON o.order_id = p.order_id

    WHERE
        o.order_purchase_timestamp >= '2017-01-01'

        AND o.order_purchase_timestamp < '2018-01-01'

    GROUP BY
        month_date,
        month
)

SELECT

    month_date,

    month,

    monthly_credit_card_volume,

    SUM(monthly_credit_card_volume)
        OVER (
            ORDER BY month_date
            ROWS BETWEEN UNBOUNDED PRECEDING
            AND CURRENT ROW
        ) AS running_credit_card_volume

FROM monthly_credit_card

ORDER BY month_date
""")

print("Created gold_credit_card_running_total_2017")


# PRODUCT ORDERS — H1 VS H2 2017

con.execute("""
CREATE OR REPLACE TABLE gold_product_orders_h1_h2_2017 AS

SELECT

    COALESCE(
        t.product_category_name_english,
        p.product_category_name,
        'Unknown'
    ) AS product_type,

    COUNT(
        DISTINCT CASE
            WHEN
                o.order_purchase_timestamp >= '2017-01-01'
                AND o.order_purchase_timestamp < '2017-07-01'
            THEN o.order_id
        END
    ) AS h1_2017_orders,

    COUNT(
        DISTINCT CASE
            WHEN
                o.order_purchase_timestamp >= '2017-07-01'
                AND o.order_purchase_timestamp < '2018-01-01'
            THEN o.order_id
        END
    ) AS h2_2017_orders,

    COUNT(
        DISTINCT o.order_id
    ) AS total_2017_orders

FROM silver_orders o

INNER JOIN silver_order_items oi
    ON o.order_id = oi.order_id

INNER JOIN silver_products p
    ON oi.product_id = p.product_id

LEFT JOIN silver_translation t
    ON p.product_category_name =
       t.product_category_name

WHERE
    o.order_purchase_timestamp >= '2017-01-01'

    AND o.order_purchase_timestamp < '2018-01-01'

GROUP BY
    product_type

ORDER BY
    total_2017_orders DESC
""")

print("Created gold_product_orders_h1_h2_2017.")


Created gold_sales_monthly_2017.
Created gold_sales_peak_2017.
Created gold_credit_card_running_total_2017
Created gold_product_orders_h1_h2_2017.


In [21]:
# Sellers and Operations

# CLOSED DEALS — Q3 2018

con.execute("""
CREATE OR REPLACE TABLE gold_closed_deals_q3_2018 AS

SELECT

    COUNT(*) AS closed_deals_q3_2018

FROM silver_closed_deals

WHERE
    won_date >= '2018-07-01'

    AND won_date < '2018-10-01'
""")

print("Created gold_closed_deals_q3_2018")


# SELLERS PER CITY BY STATE

con.execute("""
CREATE OR REPLACE TABLE gold_sellers_per_city_state AS

SELECT

    seller_state,

    COUNT(
        DISTINCT seller_id
    ) AS seller_count,

    COUNT(
        DISTINCT seller_city
    ) AS city_count,

    ROUND(
        COUNT(DISTINCT seller_id) * 1.0
        /
        NULLIF(
            COUNT(DISTINCT seller_city),
            0
        ),
        2
    ) AS sellers_per_city

FROM silver_sellers

GROUP BY
    seller_state

ORDER BY
    sellers_per_city DESC
""")

print("Created gold_sellers_per_city_state.")


# CUSTOMER COUNT BY STATE — 2018

con.execute("""
CREATE OR REPLACE TABLE gold_customers_by_state_2018 AS

SELECT

    c.customer_state,

    COUNT(
        DISTINCT c.customer_unique_id
    ) AS customer_count,

    COUNT(
        DISTINCT o.order_id
    ) AS order_count,

    CASE
        WHEN COUNT(
            DISTINCT c.customer_unique_id
        ) > 300
        THEN 'Yes'
        ELSE 'No'
    END AS over_300_customers

FROM silver_customers c

INNER JOIN silver_orders o
    ON c.customer_id = o.customer_id

WHERE
    o.order_purchase_timestamp >= '2018-01-01'

    AND o.order_purchase_timestamp < '2019-01-01'

GROUP BY
    c.customer_state

ORDER BY
    customer_count DESC
""")

print("Created gold_customers_by_state_2018.")


#  Number of states >300 customers

con.execute("""
CREATE OR REPLACE TABLE gold_states_over_300_customers_2018 AS

SELECT

    COUNT(*) AS states_over_300_customers

FROM gold_customers_by_state_2018

WHERE
    customer_count > 300
""")

print("Created gold_states_over_300_customers_2018.")


# CITY CANCELLATION + PAYMENT VOLUME

con.execute("""
CREATE OR REPLACE TABLE gold_city_cancellation_rate AS

SELECT

    c.customer_state,

    c.customer_city,

    COUNT(
        DISTINCT o.order_id
    ) AS total_orders,

    COUNT(
        DISTINCT CASE
            WHEN LOWER(o.order_status) = 'canceled'
            THEN o.order_id
        END
    ) AS canceled_orders,

    ROUND(
        COUNT(
            DISTINCT CASE
                WHEN LOWER(o.order_status) = 'canceled'
                THEN o.order_id
            END
        ) * 100.0
        /
        NULLIF(
            COUNT(DISTINCT o.order_id),
            0
        ),
        2
    ) AS cancellation_rate_pct

FROM silver_orders o

INNER JOIN silver_customers c
    ON o.customer_id = c.customer_id

GROUP BY
    c.customer_state,
    c.customer_city

HAVING
    COUNT(DISTINCT o.order_id) >= 20
""")

print("Created gold_city_cancellation_rate.")


# City payment volume 2017

con.execute("""
CREATE OR REPLACE TABLE gold_city_payment_volume_2017 AS

SELECT

    c.customer_state,

    c.customer_city,

    COUNT(
        DISTINCT o.order_id
    ) AS orders_2017,

    ROUND(
        SUM(p.payment_value),
        2
    ) AS payment_volume_2017

FROM silver_orders o

INNER JOIN silver_customers c
    ON o.customer_id = c.customer_id

INNER JOIN silver_payments p
    ON o.order_id = p.order_id

WHERE
    o.order_purchase_timestamp >= '2017-01-01'

    AND o.order_purchase_timestamp < '2018-01-01'

GROUP BY
    c.customer_state,
    c.customer_city
""")

print("Created gold_city_payment_volume_2017.")


# Top 3 lowest cancellation cities

con.execute("""
CREATE OR REPLACE TABLE gold_lowest_cancellation_cities AS

SELECT

    cc.customer_state,

    cc.customer_city,

    cc.total_orders,

    cc.canceled_orders,

    cc.cancellation_rate_pct,

    COALESCE(
        pv.orders_2017,
        0
    ) AS orders_2017,

    COALESCE(
        pv.payment_volume_2017,
        0
    ) AS payment_volume_2017

FROM gold_city_cancellation_rate cc

LEFT JOIN gold_city_payment_volume_2017 pv

    ON cc.customer_state =
       pv.customer_state

    AND cc.customer_city =
        pv.customer_city

ORDER BY

    cc.cancellation_rate_pct ASC,

    cc.total_orders DESC

LIMIT 3
""")

print("Created gold_lowest_cancellation_cities.")


# MARKETING LEADS BY ORIGIN

con.execute("""
CREATE OR REPLACE TABLE gold_leads_by_origin_2017_2018 AS

SELECT

    COALESCE(
        origin,
        'Unknown'
    ) AS origin,

    COUNT(*) AS lead_count,

    CASE
        WHEN COUNT(*) > 200
        THEN 'Over 200'
        ELSE '200 or less'
    END AS lead_group

FROM silver_leads

WHERE
    first_contact_date >= '2017-01-01'

    AND first_contact_date < '2019-01-01'

GROUP BY
    origin

ORDER BY
    lead_count DESC
""")

print("Created gold_leads_by_origin_2017_2018.")


# Number of mediums >200 leads

con.execute("""
CREATE OR REPLACE TABLE gold_origins_over_200_leads AS

SELECT

    COUNT(*) AS origins_over_200_leads

FROM gold_leads_by_origin_2017_2018

WHERE
    lead_count > 200
""")

print("Created gold_origins_over_200_leads.")


Created gold_closed_deals_q3_2018
Created gold_sellers_per_city_state.
Created gold_customers_by_state_2018.
Created gold_states_over_300_customers_2018.
Created gold_city_cancellation_rate.
Created gold_city_payment_volume_2017.
Created gold_lowest_cancellation_cities.
Created gold_leads_by_origin_2017_2018.
Created gold_origins_over_200_leads.


In [22]:
# Products and Customer Experience

# MONTHLY REVIEW SCORE — H2 2017

con.execute("""
CREATE OR REPLACE TABLE gold_review_score_h2_2017 AS

SELECT

    DATE_TRUNC(
        'month',
        o.order_purchase_timestamp
    ) AS month_date,

    STRFTIME(
        o.order_purchase_timestamp,
        '%Y-%m'
    ) AS month,

    STRFTIME(
        o.order_purchase_timestamp,
        '%B'
    ) AS month_name,

    COUNT(
        DISTINCT o.order_id
    ) AS reviewed_orders,

    ROUND(
        AVG(r.review_score),
        2
    ) AS average_review_score

FROM silver_orders o

INNER JOIN gold_order_reviews r
    ON o.order_id = r.order_id

WHERE
    o.order_purchase_timestamp >= '2017-07-01'

    AND o.order_purchase_timestamp < '2018-01-01'

GROUP BY
    month_date,
    month,
    month_name

ORDER BY
    month_date
""")

print("Created gold_review_score_h2_2017.")


# HIGH-VALUE ORDERS — H1 2018

# H1 2018 order values

con.execute("""
CREATE OR REPLACE TABLE gold_h1_2018_order_values AS

SELECT

    o.order_id,

    o.order_purchase_timestamp,

    op.total_payment_value,

    r.review_score

FROM silver_orders o

INNER JOIN gold_order_payments op
    ON o.order_id = op.order_id

LEFT JOIN gold_order_reviews r
    ON o.order_id = r.order_id

WHERE
    o.order_purchase_timestamp >= '2018-01-01'

    AND o.order_purchase_timestamp < '2018-07-01'
""")

print("✓ gold_h1_2018_order_values")


# Average H1 order value

con.execute("""
CREATE OR REPLACE TABLE gold_h1_2018_average_order_value AS

SELECT

    AVG(total_payment_value)
        AS average_order_value

FROM gold_h1_2018_order_values
""")

print("Created gold_h1_2018_average_order_value.")


# High-value order review analysis

con.execute("""
CREATE OR REPLACE TABLE gold_high_value_order_reviews_h1_2018 AS

SELECT

    CASE
        WHEN o.total_payment_value > a.average_order_value
        THEN 'Above Average'
        ELSE 'Average or Below'
    END AS order_value_segment,

    COUNT(*) AS order_count,

    ROUND(
        AVG(o.total_payment_value),
        2
    ) AS average_payment_value,

    ROUND(
        AVG(o.review_score),
        2
    ) AS average_review_score

FROM gold_h1_2018_order_values o

CROSS JOIN gold_h1_2018_average_order_value a

GROUP BY
    order_value_segment

ORDER BY
    order_value_segment
""")

print("Created gold_high_value_order_reviews_h1_2018.")


# Question-specific high-value result
# ------------------------------------------------------------

con.execute("""
CREATE OR REPLACE TABLE gold_high_value_order_review_answer AS

SELECT

    average_review_score,

    order_count,

    average_payment_value

FROM gold_high_value_order_reviews_h1_2018

WHERE
    order_value_segment = 'Above Average'
""")

print("Created gold_high_value_order_review_answer.")


# RIO DE JANEIRO SELLER PAYMENT ANALYSIS

# Identify orders containing RJ sellers

con.execute("""
CREATE OR REPLACE TABLE gold_rj_seller_orders AS

SELECT DISTINCT

    oi.order_id

FROM silver_order_items oi

INNER JOIN silver_sellers s
    ON oi.seller_id = s.seller_id

WHERE
    UPPER(
        TRIM(s.seller_state)
    ) = 'RJ'
""")

print("Created gold_rj_seller_orders.")


# Payment analysis

con.execute("""
CREATE OR REPLACE TABLE gold_rj_seller_payment_by_type AS

SELECT

    p.payment_type,

    COUNT(*) AS payment_count,

    COUNT(
        DISTINCT p.order_id
    ) AS order_count,

    ROUND(
        AVG(p.payment_value),
        2
    ) AS average_payment_value,

    ROUND(
        SUM(p.payment_value),
        2
    ) AS total_payment_value

FROM silver_payments p

INNER JOIN gold_rj_seller_orders rj
    ON p.order_id = rj.order_id

INNER JOIN gold_order_reviews r
    ON p.order_id = r.order_id

WHERE
    r.review_score >= 4

GROUP BY
    p.payment_type

ORDER BY
    average_payment_value DESC
""")

print("Created gold_rj_seller_payment_by_type.")


# ELECTRONICS AVERAGE WEIGHT

con.execute("""
CREATE OR REPLACE TABLE gold_electronics_average_weight AS

SELECT

    COUNT(*) AS electronics_product_count,

    ROUND(
        AVG(p.product_weight_g),
        2
    ) AS average_weight_g,

    ROUND(
        AVG(p.product_weight_g) / 1000.0,
        3
    ) AS average_weight_kg

FROM silver_products p

LEFT JOIN silver_translation t

    ON p.product_category_name =
       t.product_category_name

WHERE
    LOWER(
        COALESCE(
            t.product_category_name_english,
            ''
        )
    ) = 'electronics'
""")

print("Created gold_electronics_average_weight.")


Created gold_review_score_h2_2017.
✓ gold_h1_2018_order_values
Created gold_h1_2018_average_order_value.
Created gold_high_value_order_reviews_h1_2018.
Created gold_high_value_order_review_answer.
Created gold_rj_seller_orders.
Created gold_rj_seller_payment_by_type.
Created gold_electronics_average_weight.


In [23]:
#  KPI SUMMARY

con.execute("""
CREATE OR REPLACE TABLE gold_dashboard_kpis AS

SELECT

    -- Sales
    (
        SELECT total_sales
        FROM gold_sales_peak_2017
    ) AS peak_month_sales_2017,

    (
        SELECT month_name
        FROM gold_sales_peak_2017
    ) AS peak_sales_month_2017,


    -- Credit card
    (
        SELECT MAX(running_credit_card_volume)
        FROM gold_credit_card_running_total_2017
    ) AS total_credit_card_payment_volume_2017,


    -- Electronics
    (
        SELECT average_weight_kg
        FROM gold_electronics_average_weight
    ) AS average_electronics_weight_kg,


    -- Reviews
    (
        SELECT AVG(average_review_score)
        FROM gold_review_score_h2_2017
    ) AS average_review_score_h2_2017,


    -- Closed deals
    (
        SELECT closed_deals_q3_2018
        FROM gold_closed_deals_q3_2018
    ) AS closed_deals_q3_2018,


    -- Customers
    (
        SELECT states_over_300_customers
        FROM gold_states_over_300_customers_2018
    ) AS states_over_300_customers_2018,


    -- Marketing
    (
        SELECT origins_over_200_leads
        FROM gold_origins_over_200_leads
    ) AS origins_over_200_leads_2017_2018


""")

print("Created gold_dashboard_kpis.")



Created gold_dashboard_kpis.


In [24]:
#  DASHBOARD TABLE LIST

dashboard_tables = [

    # Overview
    "gold_dashboard_kpis",

    # Sales & Revenue
    "gold_sales_monthly_2017",
    "gold_sales_peak_2017",
    "gold_credit_card_running_total_2017",
    "gold_product_orders_h1_h2_2017",

    # Sellers & Operations
    "gold_closed_deals_q3_2018",
    "gold_sellers_per_city_state",
    "gold_customers_by_state_2018",
    "gold_states_over_300_customers_2018",
    "gold_city_cancellation_rate",
    "gold_city_payment_volume_2017",
    "gold_lowest_cancellation_cities",
    "gold_leads_by_origin_2017_2018",
    "gold_origins_over_200_leads",

    # Customer Experience & Products
    "gold_review_score_h2_2017",
    "gold_h1_2018_order_values",
    "gold_h1_2018_average_order_value",
    "gold_high_value_order_reviews_h1_2018",
    "gold_high_value_order_review_answer",
    "gold_rj_seller_payment_by_type",
    "gold_electronics_average_weight"
]


# EXPORT DASHBOARD TABLES TO CSV

OUTPUT_DIR = "../outputs/gold_dashboard"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

print("\n" + "=" * 60)
print("EXPORTING DASHBOARD TABLES")
print("=" * 60)

for table in dashboard_tables:

    output_path = os.path.join(
        OUTPUT_DIR,
        f"{table}.csv"
    )

    df = con.execute(
        f"SELECT * FROM {table}"
    ).fetchdf()

    df.to_csv(
        output_path,
        index=False
    )

    print(
        f"Created {table}: "
        f"{len(df):,} rows"
    )


# PRINT KPI SUMMARY

print("\n" + "=" * 60)
print("DASHBOARD KPI SUMMARY")
print("=" * 60)

kpi = con.execute("""
    SELECT *
    FROM gold_dashboard_kpis
""").fetchdf()

print(kpi.to_string(index=False))


# PRINT QUESTION RESULTS

print("\n" + "=" * 60)
print("QUESTION RESULTS")
print("=" * 60)


print("\nQ1 — Closed deals won in Q3 2018")
print(
    con.execute("""
        SELECT *
        FROM gold_closed_deals_q3_2018
    """).fetchdf()
)


print("\nQ2 — Average electronics weight")
print(
    con.execute("""
        SELECT *
        FROM gold_electronics_average_weight
    """).fetchdf()
)


print("\nQ3 — Highest sales month in 2017")
print(
    con.execute("""
        SELECT *
        FROM gold_sales_peak_2017
    """).fetchdf()
)


print("\nQ4 — Average review score H2 2017")
print(
    con.execute("""
        SELECT *
        FROM gold_review_score_h2_2017
    """).fetchdf()
)


print("\nQ5 — Sellers per city by state")
print(
    con.execute("""
        SELECT *
        FROM gold_sellers_per_city_state
    """).fetchdf()
)


print("\nQ6 — States with >300 customers")
print(
    con.execute("""
        SELECT *
        FROM gold_states_over_300_customers_2018
    """).fetchdf()
)


print("\nQ7 — Origins with >200 leads")
print(
    con.execute("""
        SELECT *
        FROM gold_origins_over_200_leads
    """).fetchdf()
)


print("\nQ8 — High-value order review score")
print(
    con.execute("""
        SELECT *
        FROM gold_high_value_order_review_answer
    """).fetchdf()
)


print("\nQ9 — Credit-card running total")
print(
    con.execute("""
        SELECT *
        FROM gold_credit_card_running_total_2017
    """).fetchdf()
)


print("\nQ10 — Lowest cancellation cities")
print(
    con.execute("""
        SELECT *
        FROM gold_lowest_cancellation_cities
    """).fetchdf()
)


print("\nQ11 — RJ seller average payment by type")
print(
    con.execute("""
        SELECT *
        FROM gold_rj_seller_payment_by_type
    """).fetchdf()
)


print("\nQ12 — H1 vs H2 product orders")
print(
    con.execute("""
        SELECT *
        FROM gold_product_orders_h1_h2_2017
    """).fetchdf()
)


# BASIC VALIDATION

print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)


# H2 should have 6 months
h2_months = con.execute("""
    SELECT COUNT(*)
    FROM gold_review_score_h2_2017
""").fetchone()[0]

print(
    f"H2 2017 review months: {h2_months}"
)

if h2_months != 6:
    print("WARNING: Expected 6 months for H2 2017.")


# 2017 credit card should have 12 months
credit_card_months = con.execute("""
    SELECT COUNT(*)
    FROM gold_credit_card_running_total_2017
""").fetchone()[0]

print(
    f"2017 credit-card months: {credit_card_months}"
)

if credit_card_months != 12:
    print("WARNING: Expected 12 months for 2017.")


# Cancellation rate
cancellation_range = con.execute("""
    SELECT
        MIN(cancellation_rate_pct),
        MAX(cancellation_rate_pct)

    FROM gold_city_cancellation_rate
""").fetchone()

print(
    f"Cancellation rate range: "
    f"{cancellation_range[0]}% - "
    f"{cancellation_range[1]}%"
)



EXPORTING DASHBOARD TABLES
Created gold_dashboard_kpis: 1 rows
Created gold_sales_monthly_2017: 12 rows
Created gold_sales_peak_2017: 1 rows
Created gold_credit_card_running_total_2017: 12 rows
Created gold_product_orders_h1_h2_2017: 73 rows
Created gold_closed_deals_q3_2018: 1 rows
Created gold_sellers_per_city_state: 23 rows
Created gold_customers_by_state_2018: 27 rows
Created gold_states_over_300_customers_2018: 1 rows
Created gold_city_cancellation_rate: 538 rows
Created gold_city_payment_volume_2017: 3,408 rows
Created gold_lowest_cancellation_cities: 3 rows
Created gold_leads_by_origin_2017_2018: 11 rows
Created gold_origins_over_200_leads: 1 rows
Created gold_review_score_h2_2017: 6 rows
Created gold_h1_2018_order_values: 41,187 rows
Created gold_h1_2018_average_order_value: 1 rows
Created gold_high_value_order_reviews_h1_2018: 2 rows
Created gold_high_value_order_review_answer: 1 rows
Created gold_rj_seller_payment_by_type: 4 rows
Created gold_electronics_average_weight: 1 ro

In [25]:
# CLOSE DATABASE

con.close()

print("\n" + "=" * 60)
print("GOLD LAYER COMPLETE")
print("=" * 60)

print(
    f"Dashboard tables exported to: "
    f"{OUTPUT_DIR}"
)


GOLD LAYER COMPLETE
Dashboard tables exported to: ../outputs/gold_dashboard
